In [1]:
import warnings
from statsmodels.tsa.stattools import adfuller
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
import sklearn.model_selection as ms
import yfinance as yf

warnings.filterwarnings("ignore", category=UserWarning)

print("==================================================")
print(" HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V5.1 - BASELINE & DIAGNOSTICS)")
print("==================================================")

# =====================================================================
# TẦNG 1: DATA & CLEANING LAYER (UTC Clock & Transparency Logs)
# =====================================================================
print("\n--- [Tầng 1] Tải dữ liệu 5 năm và làm sạch chân độc lập ---")

usd_krw = yf.download("KRW=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]
usd_vnd = yf.download("VND=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]

if isinstance(usd_krw, pd.DataFrame):
  usd_krw = usd_krw.iloc[:, 0]
if isinstance(usd_vnd, pd.DataFrame):
  usd_vnd = usd_vnd.iloc[:, 0]

# Loại bỏ Bar chưa đóng của ngày hiện tại dựa trên giờ UTC
utc_today = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
if usd_krw.index[-1] >= utc_today:
  usd_krw = usd_krw.iloc[:-1]
if usd_vnd.index[-1] >= utc_today:
  usd_vnd = usd_vnd.iloc[:-1]

df_raw = pd.DataFrame({"USD_KRW": usd_krw, "USD_VND": usd_vnd}).dropna()
initial_len = len(df_raw)

# Lọc stale data và spike trên từng chân độc lập
df = df_raw.copy()
for col in ["USD_KRW", "USD_VND"]:
  df = df[df[col].diff() != 0]
stale_dropped = initial_len - len(df)

df_clean = df.copy()
for col in ["USD_KRW", "USD_VND"]:
  df_clean = df_clean[df_clean[col].pct_change().abs() < 0.02]
spike_dropped = len(df) - len(df_clean)

print(f" -> Tổng số mẫu thô ban đầu: {initial_len}")
print(f" -> Loại bỏ do Stale Data (lặp giá): {stale_dropped} hàng")
print(f" -> Loại bỏ do Spike bất thường (>2%): {spike_dropped} hàng")

raw_prices = df_clean["USD_VND"] / df_clean["USD_KRW"]

if raw_prices.index.tz is not None:
  raw_prices.index = raw_prices.index.tz_localize(None)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()

log_prices = np.log(raw_prices)
ROUND_TRIP_COST = 0.0040  # 0.4% chi phí khứ hồi cơ sở


# =====================================================================
# TẦNG 2: OPTIMIZED FFD LAYER (ADF Test Diagnostics)
# =====================================================================
print("\n--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---")


def get_weights_ffd(d, thres=1e-4):
  w = [1.0]
  k = 1
  while True:
    w_ = -w[-1] / k * (d - k + 1)
    if abs(w_) < thres:
      break
    w.append(w_)
    k += 1
  return np.array(w[::-1])


def fractional_diff_ffd(series, d, thres=1e-4):
  w = get_weights_ffd(d, thres)
  width = len(w) - 1
  df_res = pd.Series(index=series.index, dtype=float)
  for iloc in range(width, len(series)):
    loc = series.index[iloc]
    sub_series = series.iloc[iloc - width : iloc + 1]
    df_res.loc[loc] = np.dot(w, sub_series.values)
  return df_res.dropna()


best_d = 0.05
tested_ds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
for d_test in tested_ds:
  test_series = fractional_diff_ffd(log_prices, d=d_test, thres=1e-4)
  if len(test_series) > 100:
    adf_pvalue = adfuller(test_series)[1]
    if adf_pvalue < 0.05:
      best_d = d_test
      break
    best_d = d_test

stationary_series = fractional_diff_ffd(log_prices, d=best_d, thres=1e-4)
ffd_dropped = len(log_prices) - len(stationary_series)
print(f" -> Bậc d tối ưu: {best_d} | Số hàng mất do cửa sổ FFD: {ffd_dropped}")


# =====================================================================
# TẦNG 3: FEATURE ENGINEERING & TRIPLE BARRIER METHOD (Exec t+1)
# =====================================================================
print("\n--- [Tầng 3] Xây dựng đặc trưng & Gán nhãn khớp lệnh t+1 ---")

clean_features = pd.DataFrame(index=log_prices.index, dtype=float)
clean_features["Log_Price"] = log_prices
clean_features["Frac_Diff"] = stationary_series
clean_features["Rolling_Mean"] = stationary_series.rolling(window=5).mean()
clean_features["Rolling_Std"] = stationary_series.rolling(window=5).std()
clean_features["Return_1d"] = log_prices.diff()

# Primary Model (Momentum 10 phiên)
primary_diff = log_prices.diff(10)
primary_signal = np.where(primary_diff.isna(), np.nan, np.where(primary_diff > 0, 1, -1))
clean_features["Primary_Side"] = pd.Series(primary_signal, index=log_prices.index)

daily_vol = raw_prices.pct_change().ewm(span=30).std().dropna()


def apply_triple_barrier_t1(prices_series, events, pt_sl):
  out = pd.DataFrame(index=events.index)
  out["ret"] = np.nan
  out["bin"] = 0
  for loc, t1 in events["t1"].items():
    loc_idx = prices_series.index.get_loc(loc)
    if loc_idx + 1 >= len(prices_series):
      continue
    exec_loc = prices_series.index[loc_idx + 1]

    path = prices_series.loc[exec_loc:t1]
    if len(path) <= 1:
      continue
    base_val = prices_series.loc[exec_loc]
    ret = (path / base_val) - 1
    side = events.loc[loc, "side"]
    ret_directional = ret * side
    target = max(events.loc[loc, "trgt"], 0.001)

    pt = target * pt_sl[0]
    sl = -target * pt_sl[1]

    hit_pt = ret_directional[ret_directional >= pt].index.min()
    hit_sl = ret_directional[ret_directional <= sl].index.min()

    if pd.notna(hit_pt) and (pd.isna(hit_sl) or hit_pt < hit_sl):
      out.loc[loc, "bin"] = 1
      out.loc[loc, "ret"] = ret.loc[hit_pt] * side
    elif pd.notna(hit_sl) and (pd.isna(hit_pt) or hit_sl < hit_pt):
      out.loc[loc, "bin"] = -1
      out.loc[loc, "ret"] = ret.loc[hit_sl] * side
    else:
      out.loc[loc, "bin"] = 0
      out.loc[loc, "ret"] = ret.iloc[-1] * side
  return out.dropna(subset=["ret"])


holding_period = 15
dropped_features = clean_features.dropna()
events_index = dropped_features.index[:-holding_period]
events = pd.DataFrame(index=events_index)
events["trgt"] = daily_vol.loc[events.index]
events["side"] = dropped_features.loc[events.index, "Primary_Side"]
events["t1"] = raw_prices.index[
    [
        min(
            raw_prices.index.get_loc(idx) + holding_period,
            len(raw_prices) - 1,
        )
        for idx in events.index
    ]
]

labels = apply_triple_barrier_t1(raw_prices, events, pt_sl=[2.0, 2.0])
valid_idx = labels.index.intersection(dropped_features.index)

X = dropped_features.loc[
    valid_idx,
    ["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d", "Primary_Side"],
]
meta_y = (labels.loc[valid_idx, "ret"] > 0).astype(int)

# Chẩn đoán Primary Model độc lập
primary_win_rate = (labels.loc[valid_idx, "ret"] > 0).mean()
primary_mean_ret = labels.loc[valid_idx, "ret"].mean()

print(f" -> Số lượng mẫu sạch X đưa vào Mô hình: {len(X)}")
print(f" -> Hiệu suất Primary Model đứng riêng: Hit Rate = {primary_win_rate*100:.2f}% | Lợi nhuận TB/lệnh = {primary_mean_ret*100:.4f}%")


# =====================================================================
# TẦNG 4: RIGOROUS VALIDATION (Unified Pipeline Helper & Embargo 3%)
# =====================================================================
print("\n--- [Tầng 4] Kiểm định chéo Purged CV & Hiệu chuẩn Pipeline ---")


class PurgedKFold(ms.KFold):

  def __init__(self, n_splits=5, t1=None, embargo_pct=0.03):
    super(PurgedKFold, self).__init__(
        n_splits=n_splits, shuffle=False, random_state=None
    )
    self.t1 = t1
    self.embargo_pct = embargo_pct

  def split(self, X, y=None, groups=None):
    if self.t1 is None:
      raise ValueError("Cần chuỗi t1")
    for train_index, test_index in super().split(X, y):
      test_start = X.index[test_index[0]]
      test_end = X.index[test_index[-1]]
      train_times = self.t1.loc[X.index[train_index]]

      banned_indices = train_times[
          (train_times.index <= test_end) & (train_times >= test_start)
      ].index
      train_index = np.array(
          [i for i in train_index if X.index[i] not in banned_indices]
      )

      embargo_idx = int(X.shape[0] * self.embargo_pct)
      if embargo_idx > 0:
        test_end_loc = X.index.get_loc(test_end)
        embargo_end_loc = min(test_end_loc + embargo_idx, X.shape[0])
        embargo_dates = X.index[test_end_loc + 1 : embargo_end_loc]
        train_index = np.array(
            [i for i in train_index if X.index[i] not in embargo_dates]
        )
      yield train_index, test_index


# Hàm đóng gói Pipeline hiệu chuẩn chuẩn mực dùng chung
def build_calibrated_pipeline(cv_splits):
  base_rf = RandomForestClassifier(
      n_estimators=100,
      max_depth=3,
      min_samples_leaf=30,
      random_state=42,
      class_weight="balanced",
  )
  return CalibratedClassifierCV(base_rf, method="sigmoid", cv=cv_splits)


t1_series = events["t1"].loc[X.index]
cv = PurgedKFold(n_splits=5, t1=t1_series, embargo_pct=0.03)
splits_list = list(cv.split(X))

oof_probs = np.zeros(len(X))
brier_scores = []
baseline_briers = []

for train_idx, test_idx in splits_list:
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = meta_y.iloc[train_idx], meta_y.iloc[test_idx]

  inner_cv = list(ms.KFold(n_splits=3).split(X_train))
  cal_fold = build_calibrated_pipeline(inner_cv)
  cal_fold.fit(X_train, y_train)

  probs = cal_fold.predict_proba(X_test)[:, 1]
  oof_probs[test_idx] = probs

  brier_scores.append(brier_score_loss(y_test, probs))
  p_mean = y_train.mean()
  baseline_briers.append(
      brier_score_loss(y_test, np.full_like(y_test, p_mean, dtype=float))
  )

oof_auc = roc_auc_score(meta_y, oof_probs)
print(f" -> OOF Calibrated Brier Score: {np.mean(brier_scores):.4f} (± {np.std(brier_scores):.4f})")
print(f" -> OOF Baseline Brier Score:   {np.mean(baseline_briers):.4f} (± {np.std(baseline_briers):.4f})")
print(f" -> OOF AUC-ROC Score:          {oof_auc:.4f} (Kỳ vọng ~0.50 nếu ngẫu nhiên)")

meta_model = build_calibrated_pipeline(splits_list)
meta_model.fit(X, meta_y)


# =====================================================================
# TẦNG 5: NON-OVERLAPPING OOF BACKTEST & COST SENSITIVITY
# =====================================================================
print("\n--- [Tầng 5] Backtest OOF phân tách lệnh độc lập & Live Inference ---")

historical_gains = labels.loc[labels["ret"] > 0, "ret"].abs().mean()
historical_losses = labels.loc[labels["ret"] < 0, "ret"].abs().mean()
if pd.isna(historical_gains) or historical_gains == 0:
  historical_gains = 0.01
if pd.isna(historical_losses) or historical_losses == 0:
  historical_losses = 0.01

# Hàm lọc các lệnh OOF giãn cách tối thiểu 15 phiên (không chồng lấn)
def get_non_overlapping_trades(prob_series, target_p_star, step=15):
  selected_indices = []
  i = 0
  indices = prob_series.index
  while i < len(prob_series):
    if prob_series.iloc[i] > target_p_star:
      selected_indices.append(indices[i])
      i += step  # Bỏ qua 15 phiên tiếp theo để tránh trùng lặp đợt sóng
    else:
      i += 1
  return selected_indices


print("\n[BẢNG PHÂN TÍCH ĐỘ NHẠY CẢM CHI PHÍ - ĐÃ LỌC LỆNH TRÙNG LẮP]")
print(f"{'Cost (%)':<10} | {'p* Hòa vốn (%)':<15} | {'Số lệnh Độc lập':<18} | {'Hit Rate (%)':<15} | {'Lợi nhuận TB/Lệnh':<20}")
print("-" * 85)

oof_series = pd.Series(oof_probs, index=X.index)

for c in [0.001, 0.002, 0.004, 0.008]:
  p_star = (historical_losses + c) / (historical_gains + historical_losses)
  trades_idx = get_non_overlapping_trades(oof_series, p_star, step=holding_period)
  n_trades = len(trades_idx)
  if n_trades > 0:
    trade_rets = labels.loc[trades_idx, "ret"] - c
    h_rate = (trade_rets > 0).mean() * 100
    m_ret = trade_rets.mean() * 100
    print(f"{c*100:,.1f}%      | {p_star*100:,.1f}%          | {n_trades:<18} | {h_rate:,.1f}%          | {m_ret:+,.4f}%")
  else:
    print(f"{c*100:,.1f}%      | {p_star*100:,.1f}%          | 0                  | N/A            | N/A")

# Tín hiệu Live cho phiên giao dịch mới nhất
current_data = clean_features.dropna().iloc[[-1]]
latest_date = current_data.index[0].strftime("%Y-%m-%d")
current_price = raw_prices.loc[current_data.index[0]]

pred_prob = meta_model.predict_proba(
    current_data[
        [
            "Frac_Diff",
            "Rolling_Mean",
            "Rolling_Std",
            "Return_1d",
            "Primary_Side",
        ]
    ]
)[:, 1][0]
current_side = current_data["Primary_Side"].values[0]

p_breakeven = (historical_losses + ROUND_TRIP_COST) / (historical_gains + historical_losses)
expected_value = (pred_prob * historical_gains) - ((1 - pred_prob) * historical_losses) - ROUND_TRIP_COST

print("\n" + "=" * 65)
print(f" BẢN TIN TRỰC TUYẾN BASELINE - NGÀY: {latest_date}")
print("=" * 65)
print(f" Tỷ giá KRW/VND tham chiếu: {current_price:.4f}")
print(f" Hướng đi Primary Model: {'MUA (LONG)' if current_side == 1 else 'BÁN (SHORT)'}")
print(f" Xác suất thành công (p): {pred_prob * 100:.2f}%")
print(f" Ngưỡng hòa vốn p*: {p_breakeven * 100:.2f}% (Yêu cầu Margin +3%)")
print(f" Kỳ vọng lợi nhuận thuần (EV): {expected_value * 100:.4f}%")

if expected_value > 0 and pred_prob > (p_breakeven + 0.03):
  print(" 🟢 TÍN HIỆU: ĐÈN XANH (Vượt p* + margin, đủ điều kiện giải ngân t+1)")
else:
  print(" 🔴 TÍN HIỆU: ĐÈN ĐỎ (Chưa đạt biên độ an toàn EV <= 0 / Đứng ngoài bảo toàn vốn)")
print("=" * 65)

 HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V5.1 - BASELINE & DIAGNOSTICS)

--- [Tầng 1] Tải dữ liệu 5 năm và làm sạch chân độc lập ---


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


 -> Tổng số mẫu thô ban đầu: 1299
 -> Loại bỏ do Stale Data (lặp giá): 140 hàng
 -> Loại bỏ do Spike bất thường (>2%): 14 hàng

--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---
 -> Bậc d tối ưu: 0.4 | Số hàng mất do cửa sổ FFD: 281

--- [Tầng 3] Xây dựng đặc trưng & Gán nhãn khớp lệnh t+1 ---
 -> Số lượng mẫu sạch X đưa vào Mô hình: 845
 -> Hiệu suất Primary Model đứng riêng: Hit Rate = 51.36% | Lợi nhuận TB/lệnh = 0.0919%

--- [Tầng 4] Kiểm định chéo Purged CV & Hiệu chuẩn Pipeline ---
 -> OOF Calibrated Brier Score: 0.2501 (± 0.0017)
 -> OOF Baseline Brier Score:   0.2506 (± 0.0015)
 -> OOF AUC-ROC Score:          0.4986 (Kỳ vọng ~0.50 nếu ngẫu nhiên)

--- [Tầng 5] Backtest OOF phân tách lệnh độc lập & Live Inference ---

[BẢNG PHÂN TÍCH ĐỘ NHẠY CẢM CHI PHÍ - ĐÃ LỌC LỆNH TRÙNG LẮP]
Cost (%)   | p* Hòa vốn (%)  | Số lệnh Độc lập    | Hit Rate (%)    | Lợi nhuận TB/Lệnh   
-------------------------------------------------------------------------------------
0.1%      | 51.6%     